In [1]:
import os
import rasterio
import numpy as np
from rasterio.enums import Resampling
from rasterio.warp import reproject
from tqdm import tqdm

In [2]:
# ================================
# 1. THƯ MỤC NGUỒN & ĐÍCH
# ================================
SRC_ROOT = r"E:\DownloadData\co2_ban_do\output_32648"
OUT_ROOT = r"E:\DownloadData\co2_ban_do\output_500m"
os.makedirs(OUT_ROOT, exist_ok=True)

# ================================
# 1.1. CHỈ XỬ LÝ NHỮNG FOLDER NÀY
# ================================
TARGET_FOLDERS = ["chirps", "era5", "optical_depth", "par", "smap"]   # chỉ resample các thư mục này


# ================================
# 2. HÀM RESAMPLE TO 500m
# ================================
def resample_to_500m(src_path, dst_path):

    with rasterio.open(src_path) as src:

        left, bottom, right, top = src.bounds

        new_width  = int((right - left) / 500)
        new_height = int((top - bottom) / 500)

        new_transform = rasterio.transform.from_origin(left, top, 500, 500)

        profile = src.profile.copy()
        profile.update({
            "transform": new_transform,
            "width": new_width,
            "height": new_height
        })

        with rasterio.open(dst_path, "w", **profile) as dst:

            if src.descriptions:
                for b in range(src.count):
                    dst.set_band_description(b + 1, src.descriptions[b])

            for b in range(1, src.count + 1):
                data = src.read(
                    b,
                    out_shape=(new_height, new_width),
                    resampling=Resampling.bilinear
                )
                dst.write(data.astype(src.meta["dtype"]), b)


# ================================
# 3. LOOP QUA TẤT CẢ DATASET
# ================================
for dataset in os.listdir(SRC_ROOT):

    # 🔥 Chỉ cho phép xử lý các folder nằm trong danh sách TARGET_FOLDERS
    if dataset not in TARGET_FOLDERS:
        continue

    dataset_path = os.path.join(SRC_ROOT, dataset)
    if not os.path.isdir(dataset_path):
        continue

    print("\n===============================")
    print("📌 Dataset:", dataset)
    print("===============================")

    for year_folder in os.listdir(dataset_path):

        year_path = os.path.join(dataset_path, year_folder)
        if not os.path.isdir(year_path):
            continue

        out_year_path = os.path.join(OUT_ROOT, dataset, year_folder)
        os.makedirs(out_year_path, exist_ok=True)

        tifs = [f for f in os.listdir(year_path) if f.endswith(".tif")]

        for fname in tqdm(tifs, desc=f"{dataset} {year_folder}"):
            src_file = os.path.join(year_path, fname)
            out_file = os.path.join(out_year_path, fname.replace(".tif", "_500m.tif"))
            resample_to_500m(src_file, out_file)

print("\n🎉 DONE — Chỉ những folder trong TARGET_FOLDERS đã được resample về 500m!")



📌 Dataset: chirps


chirps chirps_precipitation_2024: 100%|██████████| 31/31 [00:06<00:00,  4.90it/s]



📌 Dataset: era5


era5 era5_2024: 100%|██████████| 31/31 [01:27<00:00,  2.83s/it]



📌 Dataset: optical_depth


...


📌 Dataset: par


par par_2024: 100%|██████████| 24/24 [00:28<00:00,  1.20s/it]



📌 Dataset: smap


smap smap_2024: 100%|██████████| 31/31 [00:08<00:00,  3.61it/s]


🎉 DONE — Chỉ những folder trong TARGET_FOLDERS đã được resample về 500m!
